# softmax回归

## 分类问题

在分类问题时，使用的是one-hot编码，分量和类别是一样多的，类别对应的分量为1，其他分量为0。

## 网络架构

为了估算所有可能类别的条件概率，需要建立一个有多个输出的模型，每个类别对应一个输出。  
为了解决线性模型的分类问题，需要和输出一样多的仿射函数（Affine function）。  
我们根据四个特征三个输出为例：  
下面计算三个未规范化的预测：$o_{1}, o_{2}, o_{3}$

$
o_{1} = x_{1}w_{11} + x_{2}w_{12} + x_{3}w_{13} + x_{4}w_{14} + b_{1}\\
o_{2} = x_{1}w_{21} + x_{2}w_{22} + x_{3}w_{23} + x_{4}w_{24} + b_{2}\\
o_{3} = x_{1}w_{31} + x_{2}w_{32} + x_{3}w_{33} + x_{4}w_{34} + b_{3}
$ 


与线性回归一样，softmax回归也是一个单层神经网络，由于计算每个输出都取决于所有的输入，所以softmax回归的输出层也是全连接层\
简洁的表示就是：$ O = Wx + b$

## softmax运算

选择具有最大输出值的类别$argmax_{j}y_j$作为预测值

要把输出视为概率，必须保证任何在数据上的输出都是非负的而且总和为1。  
在分类器输出位0.5的所有样本中，我们希望这些样本刚好有一半实际上是属于预测的类别（而不是猫狗二分类去猜）。这个属性叫做校准（Calibration）。

softmax函数:\
$
\hat{y} = softmax(o) \\
\hat{y}_j = \frac{e^{o_j}}{\sum_{k}e^{o_k}}
$
在预测过程中，我们选择具有最大输出值的类别作为预测值。
softmax操作不会改变未规范化的预测o之间的大小次序，只会确定分配给每个类别的概率。在预测的过程中，仍然可以用下面的式子来选择最优可能的类别\
$argmax_{j}\hat{y}_j = argmax_{j}o_j$

## 小批量样本的矢量化

为了提高计算效率，通常会对小批量样本的数据进行矢量计算，假设读取了一个批量的样本$X$,其中特征维度（输入数量）为d， 批量大小为n。\
此外，我们假设输出中有q个类别，那么这个小批量样本的特征为$X \in \mathbb{R}^{n \times d}$，权重为$W \in \mathbb{R}^{d \times q}$，偏置为$b \in \mathbb{R}^{q}$。

softmax回归的矢量计算表达式为：\
$
O = XW + b\\
\hat{Y} = softmax(O)
$

## 损失函数

### 对数似然函数

softmax函数给出了一个向量$\hat{y}$，其中每个元素表示对应类别概率的预测值。(我们可以将其视为对给定任意输入x的每个类的条件概率)  
比如$\hat{y}_1 = P\{y=cat|x\}$

我们为了分类，最后的目标是要做最大似然：
$$P(\mathbf{Y} \mid \mathbf{X}) = \prod_{i=1}^n P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})$$

我们希望这个联合概率 $P(\mathbf{Y} \mid \mathbf{X})$ 越大越好，最好接近于 $1$（即 $100\%$ 预测正确）。这就是最大似然估计的核心思想——寻找一组模型参数，使得观测到的数据（真实标签）发生的概率最大。

前面说到，我们的目标是最大化似然概率。但在深度学习框架（如 PyTorch）中，优化器（如 SGD, Adam）被设计为最小化一个损失函数 (Loss)。数学上，最大化一个函数，等价于最小化它的负数。  
$$-\log P(\mathbf{Y} \mid \mathbf{X}) = \sum_{i=1}^n -\log P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)}) = \sum_{i=1}^n l(\mathbf{y}^{(i)}, \mathbf{\hat{y}}^{(i)})$$

在数学形式上来说，交叉熵和最大似然估计的-log函数是一致的，因为根据信息论，越不可能事件发生，信息量就越大。

交叉熵如下：  
$$l(\mathbf{y}, \mathbf{\hat{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j$$

## softmax及其导数

我们已知交叉熵损失 $l = - \sum_{j=1}^q y_j \log \hat{y}_j$，并且 Softmax 的输出 $\hat{y}_j = \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)}$

将 $\hat{y}_j$ 代入后，利用对数的除法性质 $\log(\frac{A}{B}) = \log A - \log B$，得到：$$-\sum_{j=1}^q y_j \left( \log(\exp(o_j)) - \log\sum_{k=1}^q \exp(o_k) \right)$$

化简可以得到：
$$\sum_{j=1}^q y_j \log\sum_{k=1}^q \exp(o_k) - \sum_{j=1}^q y_j o_j$$

对未经过规范化的预测 $o_j$求导可以得到，第一项：
$$\frac{\partial}{\partial o_j} \left( \log\sum_{k=1}^q \exp(o_k) \right) = \frac{1}{\sum_{k=1}^q \exp(o_k)} \cdot \exp(o_j)$$
\
最终得到的梯度公式为：
$$\partial_{o_j} l(\mathbf{y}, \mathbf{\hat{y}}) = \text{softmax}(\mathbf{o})_j - y_j$$


交叉熵$H(P, Q)$

主观概率为Q的观察者在看到根据概率P生成的数据时的预期惊讶程度